# MACCO reproduction in Google Colab

## 1 — Open GitHub first

Open **[the official MACCO repository](https://github.com/hiker-lw/MACCO)** before running any code. Read README → inspect `eval/evaluate_comp_benchmark.py` → inspect `eval/evaluate_aro_order.py` and `eval/evaluate_vl_checklist.py` → inspect `src/config/MACCO_CLIP.yaml` for the later training stage.

**Goal:** obtain the paper's Table 1 scores with the released **MACCO-CLIP ViT-B/32** checkpoint, then reproduce training. A checkpoint is a saved trained model. Running evaluation does not repeat training.

**First working milestone:** complete Sugar-Crepe on original CLIP and released MACCO, compare Relation and Attribute, and save the logs. Then add ARO, What's-up, VL-Checklist and VALSE. Download one benchmark at a time.

This notebook was prepared against Git commit `97fbae1416e30eba84cc0b8209d9cb6bdedfe6ae` on 13 October 2026. It has been checked against source files and its code syntax has been validated. **Full dataset evaluation and a Colab GPU run have not yet been performed. No reproduced scores are claimed.** OpenCLIP's version is not specified upstream; this guide uses 2.32.0 as an explicit starting choice, not a verified author version.

Run cells one at a time. Do not use Run all: later sections intentionally require data checks and a restart after installing packages.


## 2 — Open this notebook in Colab and select a GPU

In [Google Colab](https://colab.research.google.com/), choose **File → Upload notebook**, select this `.ipynb`, and save a copy in your Drive. Use **Runtime → Change runtime type → GPU**. A T4 is a reasonable starting GPU for small-batch evaluation, subject to the available runtime; A100 is preferable for later training. Availability is not guaranteed.

**Check:** the following cell should show a GPU. `/content` files can disappear when the Colab session ends, so logs and checkpoints are saved in Drive. Large extracted image folders stay on the runtime disk for evaluation. [Colab FAQ](https://research.google.com/colaboratory/faq.html)


In [ ]:
import sys, subprocess, shutil
print('Python:', sys.version)
subprocess.run(['nvidia-smi'], check=True)
print('Free runtime disk GB:', round(shutil.disk_usage('/content').free / 1e9, 1))


## 3 — Clone and pin the repository

Pinning means selecting a specific code version so a future GitHub update does not silently change your experiment. Preserve upstream files; our Colab copies will be separate. If this directory already exists, this cell verifies its commit instead of overwriting your work.


In [ ]:
from pathlib import Path
import subprocess
REPO = Path('/content/MACCO')
COMMIT = '97fbae1416e30eba84cc0b8209d9cb6bdedfe6ae'
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/hiker-lw/MACCO.git', str(REPO)], check=True)
    subprocess.run(['git', 'checkout', COMMIT], cwd=REPO, check=True)
actual_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
assert actual_commit == COMMIT, f'Unexpected commit: {actual_commit}'
print('Pinned repository:', actual_commit)
print((REPO / 'requirements.txt').read_text())


## 4 — Install evaluation packages then restart

The upstream full requirements list includes training and unrelated evaluation tools and omits `open_clip_torch`. This cell installs a smaller evaluation environment, using upstream Torch/Torchvision versions plus an explicit OpenCLIP version. It is a Colab adaptation, not proof that the author's complete environment is identical.

**After this cell succeeds, choose Runtime → Restart session.** Then rerun cells 2 and 3, skip this installation cell, and continue with cell 5. If installation fails, keep the error text and fix that before downloading data. Do not change package versions repeatedly in the same live kernel.


In [ ]:
import sys, subprocess
packages = [
    'torch==2.6.0', 'torchvision==0.21.0', 'open_clip_torch==2.32.0',
    'timm==1.0.15', 'easydict==1.13', 'numpy==2.2.3', 'pandas==2.2.3',
    'pillow==11.1.0', 'tqdm==4.67.1', 'ftfy==6.3.1',
    'huggingface_hub==0.30.2', 'spacy==3.8.4', 'nltk==3.9.1'
]
subprocess.run([sys.executable, '-m', 'pip', 'install', *packages], check=True)
print('INSTALL COMPLETE — restart the session now, then skip this cell.')


## 5 — Connect Drive and record the environment

Use a distinct run folder for every experiment. This stores versions, source changes, logs and CSV results. A seed controls some random choices; it does not guarantee identical results across every GPU/software setup.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch, open_clip, numpy as np, pandas as pd
import random, json, datetime, importlib.metadata as metadata
from pathlib import Path
import subprocess, sys, os
REPO = Path('/content/MACCO')
DATA = REPO / 'datasets'
DATA.mkdir(exist_ok=True)
PROJECT = Path('/content/drive/MyDrive/MACCO_Reproduction')
RUN = PROJECT / 'runs' / datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M%S')
RUN.mkdir(parents=True, exist_ok=False)
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
DEVICE = 'cuda'
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
env = {
    'commit': subprocess.check_output(['git','rev-parse','HEAD'], cwd=REPO, text=True).strip(),
    'python': sys.version, 'gpu': torch.cuda.get_device_name(0),
    'torch': torch.__version__, 'cuda': torch.version.cuda, 'seed': SEED,
    'packages': {p: metadata.version(p) for p in ['open_clip_torch','torchvision','numpy','pandas','spacy','nltk']},
    'evaluation_batch_size': 64, 'num_workers': 2,
    'precision': 'float32 without autocast', 'status': 'not yet evaluated'
}
(RUN / 'environment.json').write_text(json.dumps(env, indent=2))
(RUN / 'pip_freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'], text=True))
print('Results will be saved to:', RUN)


## 6 — Download the correct released checkpoint

Open [the model files](https://huggingface.co/hiker-lw/MACCO/tree/main). For the main Table 1, use `MACCO-CLIP-ViT-B-32.pt` (listed at about 605 MB). Do not use B/16, L/14 or SigLIP and compare them with B/32 targets. The exact Hugging Face revision and checkpoint hash are saved.


In [ ]:
from huggingface_hub import HfApi, hf_hub_download
import hashlib, shutil
api = HfApi()
model_revision = api.model_info('hiker-lw/MACCO').sha
dataset_revision = api.dataset_info('hiker-lw/VL-Compositionality-Benchmarks').sha
checkpoint_name = 'MACCO-CLIP-ViT-B-32.pt'
saved_ckpt = PROJECT / 'checkpoints' / model_revision / checkpoint_name
saved_ckpt.parent.mkdir(parents=True, exist_ok=True)
if not saved_ckpt.exists():
    downloaded = hf_hub_download('hiker-lw/MACCO', checkpoint_name, revision=model_revision)
    shutil.copy2(downloaded, saved_ckpt)
CHECKPOINT = REPO / checkpoint_name
shutil.copy2(saved_ckpt, CHECKPOINT)
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8*1024*1024), b''): h.update(chunk)
    return h.hexdigest()
env.update(model_revision=model_revision, dataset_revision=dataset_revision,
           checkpoint_sha256=sha256_file(CHECKPOINT))
(RUN / 'environment.json').write_text(json.dumps(env, indent=2))
print(CHECKPOINT, 'bytes:', CHECKPOINT.stat().st_size)


## 7 — Check model loading before downloading benchmarks

The following uses the same model name and local-checkpoint loading pattern as the authors' evaluation scripts. Keep the preprocessing and tokenization unchanged for comparison. A warning about QuickGELU is meaningful: OpenCLIP distinguishes GELU and QuickGELU activation functions. If results differ, record the activation and investigate it as a controlled configuration change; do not silently swap models until the score looks right. [OpenCLIP documentation](https://pypi.org/project/open-clip-torch/2.32.0/)


In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained=str(CHECKPOINT), device=DEVICE)
model.eval()
tokens = open_clip.tokenize(['a red cup beside a blue plate'])
with torch.no_grad():
    text_features = model.encode_text(tokens.to(DEVICE))
assert text_features.shape == (1, 512)
assert torch.isfinite(text_features).all()
env['activation_types'] = sorted({type(m).__name__ for m in model.modules()
                                  if 'GELU' in type(m).__name__})
env['preprocess'] = str(preprocess)
(RUN / 'environment.json').write_text(json.dumps(env, indent=2))
print('Checkpoint loads. Text output:', tuple(text_features.shape))
print('Activation:', env['activation_types'])
del model
torch.cuda.empty_cache()


## 8 — Download only Sugar-Crepe first

Open [benchmark files](https://huggingface.co/datasets/hiker-lw/VL-Compositionality-Benchmarks/tree/main). The full collection is listed at roughly 88.8 GB; do not download the entire collection for the first milestone. Sugar-Crepe is listed at about 820 MB compressed. The first archive helper below can be reused for other individual archives. Extraction needs additional runtime disk space. It preserves archive layout and prints locations to inspect.


In [ ]:
import tarfile
def download_archive(filename):
    archive = Path(hf_hub_download(
        'hiker-lw/VL-Compositionality-Benchmarks', filename,
        repo_type='dataset', revision=dataset_revision))
    print('Archive:', archive, 'GB:', round(archive.stat().st_size/1e9, 3))
    with tarfile.open(archive) as tf:
        print('First archive entries:', tf.getnames()[:12])
        tf.extractall(DATA, filter='data')
    (RUN / (filename.replace('/','_') + '.sha256.txt')).write_text(sha256_file(archive))
    return archive

def find_root(marker, expected):
    target = DATA / expected
    if (target / marker).is_file(): return target
    candidates = list(DATA.rglob(marker))
    assert len(candidates) == 1, f'Expected one {marker}, found {candidates}. Inspect extracted layout.'
    source = candidates[0].parent
    if target.exists():
        raise RuntimeError(f'{target} exists but is not the expected root. Inspect before moving files.')
    target.parent.mkdir(parents=True, exist_ok=True)
    target.symlink_to(source, target_is_directory=True)
    print('Linked dataset root:', target, '->', source)
    return target

download_archive('sugar-crepe.tar.gz')
find_root('replace_rel.json', 'sugar-crepe')
assert (DATA / 'sugar-crepe' / 'val2017').is_dir(), 'Missing val2017 images; inspect archive structure.'


## 9 — Make documented Colab copies of evaluation scripts

The original scripts remain unchanged. Adapted copies reduce evaluation batches to 64 and workers to 2, replace dataset paths, exclude Winoground from Table 1 runs, and expose individual benchmark calls. Lower evaluation batch size is a memory accommodation; it does not change the dataset or scoring rules.

Source issues recorded at the inspected commit:
- ARO Attribute points to `ARO_Relation_dataset` whereas Relation uses `ARO`.
- What's-up uses an author-local absolute path; its second VG-QA call mistakenly uses `COCO_QA`.
- VALSE annotations and the VL-Checklist Object path use `../MACCO_CLIP/...`.
- The main script runs Winoground, which is not a Table 1 column.

The What's-up correction is an explicit interpretation of the intended VG-QA evaluation, not a verified patch from the authors. Preserve both original code and the diff when reporting results.


In [ ]:
import ast, re, difflib, importlib.util
ADAPTED = REPO / 'colab_eval'
ADAPTED.mkdir(exist_ok=True)
def checked_replace(text, old, new, count=1):
    assert text.count(old) == count, f'Upstream source changed: {old!r}'
    return text.replace(old, new)
for filename in ['evaluate_comp_benchmark.py','evaluate_aro_order.py','evaluate_vl_checklist.py']:
    original = (REPO / 'eval' / filename).read_text()
    changed = re.sub(r'batch_size=(256|1024)', 'batch_size=64', original)
    changed = re.sub(r'num_workers=(20|8)', 'num_workers=2', changed)
    changed = changed.replace('../MACCO_CLIP/datasets/', str(DATA) + '/')
    changed = changed.replace('../datasets/ARO_Relation_dataset', str(DATA / 'ARO'))
    changed = changed.replace('../datasets/', str(DATA) + '/')
    changed = changed.replace('"../datasets"', repr(str(DATA)))
    if filename == 'evaluate_comp_benchmark.py':
        changed = checked_replace(changed,
            '/home/liwei/code/MACCO_CLIP/datasets/whats_up_data', str(DATA/'whats_up_data'))
        changed = checked_replace(changed,
            "vg_qa_dataset = COCO_QA(image_preprocess=image_preprocess, root_dir=dataset_dir, subset='two')",
            "vg_qa_dataset = VG_QA(image_preprocess=image_preprocess, root_dir=dataset_dir, subset='two')")
        # Evaluate only the VALSE subsets used in its Table 1 relation score.
        tree = ast.parse(changed)
        fn = next(n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name=='evaluate_valse')
        assignment = next(n for n in fn.body if isinstance(n, ast.Assign) and
                          any(isinstance(t, ast.Name) and t.id=='dataset_name_list' for t in n.targets))
        lines = changed.splitlines(keepends=True)
        lines[assignment.lineno-1:assignment.end_lineno] = [
            '    dataset_name_list = ["actant-swap.json", "action-replacement.json", "relations.json"]\n']
        changed = ''.join(lines)
        # Save each What's-up group and report equal-weight mean of three groups.
        for label, variable in [('What\'s up', 'controlled_mean'), ('COCO_QA','coco_mean'), ('VG_QA','vg_mean')]:
            needle = f'    print(f"{label} Macro Accuracy: {{(individual_accuracy_1 + individual_accuracy_2)/2:.3f}}")'
            changed = checked_replace(changed, needle,
                needle + f'\n    {variable} = (individual_accuracy_1 + individual_accuracy_2)/2')
        tree = ast.parse(changed)
        fn = next(n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name=='evaluate_whats_up')
        lines = changed.splitlines(keepends=True)
        lines.insert(fn.end_lineno,
            '    print(f"TABLE1_WHATSUP_MEAN: {(controlled_mean + coco_mean + vg_mean)/3:.6f}")\n')
        changed = ''.join(lines)
    ast.parse(changed)
    (ADAPTED / filename).write_text(changed)
    (RUN / (filename + '.diff')).write_text(''.join(difflib.unified_diff(
        original.splitlines(True), changed.splitlines(True), fromfile='upstream/'+filename,
        tofile='colab/'+filename)))
    print('Prepared:', filename)
sys.path.insert(0, str(REPO/'eval'))  # needed for the authors' bundled ARO utilities
def load_module(name, filename):
    spec = importlib.util.spec_from_file_location(name, ADAPTED / filename)
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
    return mod
comp = load_module('colab_comp', 'evaluate_comp_benchmark.py')
print('Imported main evaluation module; no benchmark has run yet.')


## 10 — Check all Sugar-Crepe files before scoring

A missing or corrupt image must stop the run. Do not silently skip examples and compare the shortened dataset with the paper. This check opens each referenced image once. It is slower than checking names alone, but prevents a partial benchmark result from looking complete.


In [ ]:
from PIL import Image
def check_image_paths(paths):
    paths = sorted(set(map(str, paths)))
    missing = [p for p in paths if not Path(p).is_file()]
    assert not missing, f'{len(missing)} missing images; examples: {missing[:5]}'
    for p in paths:
        with Image.open(p) as image: image.verify()
    print('Verified unique image files:', len(paths))

sugar_files = ['add_att.json','replace_att.json','swap_att.json','replace_rel.json',
               'add_obj.json','replace_obj.json','swap_obj.json']
counts = {}
all_paths = []
for filename in sugar_files:
    ds = comp.SugarCrepe(None, str(DATA/'sugar-crepe'), filename)
    counts[filename] = len(ds)
    all_paths.extend(x['image_path'] for x in ds.dataset)
check_image_paths(all_paths)
(RUN/'sugar_dataset_counts.json').write_text(json.dumps(counts, indent=2))
print(counts)


## 11 — Run the original CLIP baseline then MACCO

The baseline checks that the evaluation pipeline is working. The author functions load a model for each benchmark and preserve their scoring and grouping. `openai` loads original CLIP weights; the local `.pt` loads MACCO.

First run a small test on a few examples to catch loading/runtime errors. A small test is NOT a paper score. Then run the full function. Logs stream into the cell and are saved to Drive. A completed full run must include all seven Sugar-Crepe subset outputs and the Attribute/Relation aggregates.


In [ ]:
import io, contextlib, time, gc
class Tee:
    def __init__(self, *streams): self.streams=streams
    def write(self, s):
        for stream in self.streams: stream.write(s)
        return len(s)
    def flush(self):
        for stream in self.streams: stream.flush()
def run_logged(label, fn, pretrained):
    path = RUN / (label + '.txt')
    assert not path.exists(), 'Log exists; use a new label or a new run folder.'
    start = time.time()
    with path.open('w') as log:
        with contextlib.redirect_stdout(Tee(sys.stdout, log)):
            print('Function:', fn.__name__, 'weights:', pretrained)
            fn(pretrained_path=pretrained, device=DEVICE)
            print('Elapsed seconds:', round(time.time()-start, 1))
            print('RUN_COMPLETED')
    gc.collect(); torch.cuda.empty_cache()
    return path

# Diagnostic only: score the first eight relation examples without claiming a paper score.
smoke_model, _, smoke_preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained=str(CHECKPOINT), device=DEVICE)
smoke_model.eval()
smoke_ds = comp.SugarCrepe(smoke_preprocess, str(DATA/'sugar-crepe'), 'replace_rel.json')
smoke_loader = torch.utils.data.DataLoader(
    torch.utils.data.Subset(smoke_ds, range(min(8,len(smoke_ds)))),
    batch_size=8, num_workers=0, shuffle=False)
smoke_scores = comp.CLIPWrapper(smoke_model, DEVICE).get_retrieval_scores_batched(smoke_loader)
assert smoke_scores.shape == (min(8,len(smoke_ds)),1,2)
assert np.isfinite(smoke_scores).all()
print('Small diagnostic passed; beginning FULL benchmark runs.')
del smoke_model, smoke_ds, smoke_loader
gc.collect(); torch.cuda.empty_cache()

run_logged('CLIP_SugarCrepe', comp.evaluate_sugar_crepe, 'openai')
run_logged('MACCO_SugarCrepe', comp.evaluate_sugar_crepe, str(CHECKPOINT))


## 12 — Compare the first two Table 1 numbers

The script prints fractions. Multiply by 100 to compare with the table: **0.771 → 77.1**, not 0.771 versus 77.1. Attribute is the unweighted mean of Add/Replace/Swap Attribute; Relation is Replace Relation. Object results are useful diagnostics but are not the Sugar-Crepe columns in Table 1.

Expected Table 1 entries: CLIP Relation 68.8, Attribute 70.8; MACCO Relation 77.1, Attribute 79.1. Table 18 lists detailed subset scores and has CLIP Replace Relation 68.9, a 0.1-point inconsistency with Table 1. Preserve both references; do not tune code to eliminate a paper rounding/reporting inconsistency.

There is no universal accepted tolerance specified in the paper for released-weight reproduction. Record every difference. A training rerun has additional seed variability (Table 15); this does not excuse unexplained differences when reevaluating the same released weights.


In [ ]:
PAPER_CLIP = {'ARO Relation':58.7,'ARO Attribute':62.7,'ARO Order':54.1,
              'SugarCrepe Relation':68.8,'SugarCrepe Attribute':70.8,
              'VL Checklist Relation':63.6,'VL Checklist Attribute':67.7,
              'VALSE Relation':70.1, 'Whats up':41.8}
PAPER_MACCO = {'ARO Relation':73.1,'ARO Attribute':68.5,'ARO Order':76.0,
               'SugarCrepe Relation':77.1,'SugarCrepe Attribute':79.1,
               'VL Checklist Relation':70.2,'VL Checklist Attribute':68.7,
               'VALSE Relation':75.3, 'Whats up':43.2}
rows = []
def record(model_name, metric, fraction):
    assert 0 <= fraction <= 1, 'Expected accuracy fraction, not percentage.'
    expected = (PAPER_CLIP if model_name=='CLIP' else PAPER_MACCO)[metric]
    rows.append({'model':model_name,'metric':metric,'paper':expected,
                 'ours':100*fraction,'difference_pp':100*fraction-expected})
def get_number(log_text, pattern):
    matches = re.findall(pattern, log_text)
    assert len(matches)==1, f'Expected exactly one aggregate for {pattern}, got {matches}'
    return float(matches[0])
for model_name in ['CLIP','MACCO']:
    text = (RUN/f'{model_name}_SugarCrepe.txt').read_text()
    assert 'RUN_COMPLETED' in text
    for kind in ['Relation','Attribute']:
        value = get_number(text, rf'sugar_crepe {kind} Average Accuracy: ([0-9.]+)')
        record(model_name, 'SugarCrepe '+kind, value)
comparison = pd.DataFrame(rows)
comparison.to_csv(RUN/'table1_comparison.csv', index=False)
display(comparison)


## 13 — Decide whether the first milestone is working

You are on the right track when imports and checkpoint loading succeed, all referenced images are present, both full runs complete, and the results can be compared with Table 1. Investigate differences before adding more benchmarks.

If both models differ substantially, check data, preprocessing, tokenizer, score aggregation and package versions. If only MACCO differs, check the checkpoint/model pairing and activation configuration. Preserve source diffs and logs. Do not change the dataset, silently skip images, train extra epochs, or change scoring just to move the number toward the table.

**Save the Colab notebook with outputs in Drive now.** The CSV and logs are already saved there. Send Sir the measured results, not just the expected scores.


## 14 — Add ARO Relation and Attribute

Only continue after the first milestone. Download `ARO.tar.gz` (listed at about 5.41 GB compressed). Relation uses a category-level mean with the authors' exclusions; Attribute uses categories with at least 25 examples. Keep their evaluator rules rather than replacing them with a pooled percentage.


In [ ]:
# OPTIONAL — run after Sugar-Crepe.
download_archive('ARO.tar.gz')
find_root('visual_genome_relation.json', 'ARO')
assert (DATA/'ARO'/'visual_genome_attribution.json').is_file()
for cls in [comp.VG_Relation, comp.VG_Attribution]:
    ds = cls(image_preprocess=None, root_dir=str(DATA/'ARO'))
    print(cls.__name__, 'examples:', len(ds))
    check_image_paths([x['image_path'] for x in ds.dataset])
for model_name, weights in [('CLIP','openai'),('MACCO',str(CHECKPOINT))]:
    for kind, fn in [('Relation',comp.evaluate_aro_relation),('Attribute',comp.evaluate_aro_attribute)]:
        path=run_logged(f'{model_name}_ARO_{kind}',fn,weights)
        label='VG-Relation' if kind=='Relation' else 'VG-Attribution'
        value=get_number(path.read_text(),rf'{label} Macro Accuracy: ([0-9.]+)')
        record(model_name, 'ARO '+kind, value)
pd.DataFrame(rows).to_csv(RUN/'table1_comparison.csv',index=False)
display(pd.DataFrame(rows))


## 15 — Add ARO Order after checking its extra requirements

Open `eval/evaluate_aro_order.py` again. It creates four shuffled incorrect captions for each correct caption using spaCy and NLTK. That randomness affects repeatability, so reset the same seed before each model and save counts. You need `coco_karpathy_test.json`, `flickr30k_test.json`, the COCO images referenced by the first file under `datasets/coco/`, and Flickr30k images under `datasets/ARO/flickr30k-images/`.

Do not assume the ARO archive contains every COCO image. The next cell checks the actual filenames in the annotation files. If missing, obtain the referenced images following the dataset providers' instructions before running scores. Do not replace COCO 2014 with 2017 images or guess a test split.

ARO Order = average of COCO-Order and Flickr-Order. Table 17 MACCO targets: 72.3 and 79.6, whose average rounds to 76.0 in Table 1.


In [ ]:
# OPTIONAL — install language resources, then check the actual image layout.
subprocess.run([sys.executable,'-m','spacy','download','en_core_web_sm'],check=True)
import nltk
assert nltk.download('punkt')
assert nltk.download('punkt_tab')
order = load_module('colab_order','evaluate_aro_order.py')
root = DATA/'ARO'
for filename in ['coco_karpathy_test.json','flickr30k_test.json']:
    assert (root/filename).is_file(), f'Missing {root/filename}'
coco_ann=json.loads((root/'coco_karpathy_test.json').read_text())
flickr_ann=json.loads((root/'flickr30k_test.json').read_text())
check_image_paths([DATA/'coco'/x['image'] for x in coco_ann])
check_image_paths([root/'flickr30k-images'/x['image'] for x in flickr_ann])
for model_name, weights in [('CLIP','openai'),('MACCO',str(CHECKPOINT))]:
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    a=run_logged(f'{model_name}_COCO_Order',order.evaluate_aro_coco_order,weights)
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    b=run_logged(f'{model_name}_Flickr_Order',order.evaluate_aro_flickr30k_order,weights)
    a_value=get_number(a.read_text(),r'COCO_Order Accuracy: ([0-9.]+)')
    b_value=get_number(b.read_text(),r'Flickr30k_Order Accuracy: ([0-9.]+)')
    record(model_name,'ARO Order',(a_value+b_value)/2)
pd.DataFrame(rows).to_csv(RUN/'table1_comparison.csv',index=False)
display(pd.DataFrame(rows))


## 16 — Add What's-up and verify the aggregation

Download `whats_up_data.tar.gz` (listed at about 1.67 GB compressed). Do not compare only controlled images with Table 1's 43.2. Table 20 reports MACCO controlled What's-up 34.2, COCO-spatial 47.1 and GQA-spatial 48.3; their equal-weight mean is 43.2. Each group combines its two subsets.

The adapted script corrects the second VG-QA dataset class and prints the final mean. Report that correction alongside your results, because the released script as written cannot be assumed to match the intended evaluation. If the group scores disagree with Table 20, inspect the two-subset aggregation and the upstream benchmark protocol before claiming reproduction.


In [ ]:
# OPTIONAL — verify every image referenced by the six dataset subsets.
download_archive('whats_up_data.tar.gz')
find_root('controlled_images_dataset.json','whats_up_data')
root=DATA/'whats_up_data'
check_image_paths([root/x['image_path'] for filename in
                  ['controlled_images_dataset.json','controlled_clevr_dataset.json']
                  for x in json.loads((root/filename).read_text())])
for name in ['one','two']:
    ann=json.loads((root/f'coco_qa_{name}_obj.json').read_text())
    check_image_paths([root/'val2017'/f'{int(x[0]):012d}.jpg' for x in ann])
    ann=json.loads((root/f'vg_qa_{name}_obj.json').read_text())
    check_image_paths([root/'vg_images'/f'{x[0]}.jpg' for x in ann])
for model_name,weights in [('CLIP','openai'),('MACCO',str(CHECKPOINT))]:
    path=run_logged(f'{model_name}_WhatsUp',comp.evaluate_whats_up,weights)
    value=get_number(path.read_text(),r'TABLE1_WHATSUP_MEAN: ([0-9.]+)')
    record(model_name,'Whats up',value)
pd.DataFrame(rows).to_csv(RUN/'table1_comparison.csv',index=False)
display(pd.DataFrame(rows))


## 17 — Plan the large VL-Checklist download before proceeding

Open [its files](https://huggingface.co/datasets/hiker-lw/VL-Compositionality-Benchmarks/tree/main/VL_checklist). Inspect the available names and sizes instead of guessing archive filenames. The repository README describes multipart HAKE, SWIG and VG archives: concatenate numbered parts in order before extraction. Preserve filenames and extract so both folders exist under `datasets/VL_checklist/`: `VL_checklist_datasets/` and `VL_checklist_json_data/`. Check available Colab disk before downloading.

For Table 1 you need Relation and Attribute. The author function also runs Object afterward; that is a separate result, so the cell below excludes that block in another documented copy. Relation averages the three action results, then averages that action mean with the spatial result. Attribute averages VAW/VG scores by attribute type, then averages the five types. Keep these rules.


In [ ]:
# OPTIONAL — inventory only; this does not download the large collection.
vl_files=[f for f in api.list_repo_files('hiker-lw/VL-Compositionality-Benchmarks',
          repo_type='dataset',revision=dataset_revision) if f.startswith('VL_checklist/')]
print('\n'.join(vl_files))
print('Free runtime disk GB:',round(shutil.disk_usage('/content').free/1e9,1))


**Manual data gate:** download the required parts from the listed revision, join matching parts in filename order, extract them, and inspect the folder structure. Do not run the next cell until that is done. Files missing from a dataset archive are a data issue to resolve, not examples to skip.


In [ ]:
# OPTIONAL — after preparing VL-Checklist data.
vl_source=(ADAPTED/'evaluate_vl_checklist.py').read_text()
start=vl_source.index('    # evaluate on VL_checklist Object')
end=vl_source.index('class CLIPWrapper:')
vl_table1=vl_source[:start]+vl_source[end:]
ast.parse(vl_table1)
(ADAPTED/'vl_table1.py').write_text(vl_table1)
(RUN/'vl_table1_scope.diff').write_text(''.join(difflib.unified_diff(
    vl_source.splitlines(True),vl_table1.splitlines(True))))
vl=load_module('colab_vl','vl_table1.py')
root=DATA/'VL_checklist'
paths=[]
for name in ['hake_action.json','swig_action.json','vg_action.json','vg_spatial.json']:
    ds=vl.VL_checklist_Relation(None,str(root),name)
    paths.extend(x['image_path'] for x in ds.dataset)
for group in ['vaw','vg']:
    for kind in ['action','color','material','size','state']:
        ds=vl.VL_checklist_Attribute(None,str(root),f'{group}/{kind}.json')
        paths.extend(x['image_path'] for x in ds.dataset)
check_image_paths(paths)
for model_name,weights in [('CLIP','openai'),('MACCO',str(CHECKPOINT))]:
    path=run_logged(f'{model_name}_VLChecklist',vl.evaluate_vl_checklist,weights)
    for kind in ['Relation','Attribute']:
        value=get_number(path.read_text(),rf'VL_checklist_{kind} Average Accuracy: ([0-9.]+)')
        record(model_name,'VL Checklist '+kind,value)
pd.DataFrame(rows).to_csv(RUN/'table1_comparison.csv',index=False)
display(pd.DataFrame(rows))


## 18 — Add the Table 1 VALSE subsets

Download `VALSE.tar.gz` (listed at about 2.24 GB compressed). The author code's Table 1 Relation aggregate uses two action files plus `relations.json`. Action images depend on VL-Checklist's HAKE/SWIG/VG image folders; spatial-relation images use Sugar-Crepe's image folder. Downloading the VALSE archive alone is not sufficient.

The adapted function runs only those three subsets rather than unrelated VALSE tasks requiring additional datasets. Table 20 MACCO targets: Action 78.6, spatial Relation 72.0, mean 75.3.


In [ ]:
# OPTIONAL — after VL-Checklist and Sugar-Crepe images are ready.
download_archive('VALSE.tar.gz')
find_root('actant-swap.json','VALSE/probing_data')
paths=[]
for name in ['actant-swap.json','action-replacement.json','relations.json']:
    root=DATA/'sugar-crepe' if name=='relations.json' else DATA/'VL_checklist'/'VL_checklist_datasets'
    ds=comp.VALSE(None,str(root),name)
    print(name,'examples:',len(ds))
    paths.extend(x['image_path'] for x in ds.dataset)
check_image_paths(paths)
for model_name,weights in [('CLIP','openai'),('MACCO',str(CHECKPOINT))]:
    path=run_logged(f'{model_name}_VALSE',comp.evaluate_valse,weights)
    value=get_number(path.read_text(),r'VALSE <Relation> Average Accuracy: ([0-9.]+)')
    record(model_name,'VALSE Relation',value)
pd.DataFrame(rows).to_csv(RUN/'table1_comparison.csv',index=False)
display(pd.DataFrame(rows))


## 19 — Report the completed Table 1 reproduction

Required evidence: all nine columns with paper/your/difference values; exact repository commit; checkpoint hash; model and dataset revisions; software/GPU details; full logs; sample counts; and every source change. If any metric is unavailable, label it **not run**. Do not fill it with the expected value. Read the final CSV and save this notebook with outputs.

Table 1's MACCO row is 73.1, 68.5, 76.0, 77.1, 79.1, 70.2, 68.7, 75.3, 43.2 in the paper's column order. These are targets, not this notebook's outputs.


In [ ]:
frame=pd.DataFrame(rows)
frame=frame.drop_duplicates(['model','metric'],keep='last')
expected=pd.DataFrame([{'model':model,'metric':metric,'paper':value}
    for model,targets in [('CLIP',PAPER_CLIP),('MACCO',PAPER_MACCO)]
    for metric,value in targets.items()])
report=expected.merge(frame[['model','metric','ours','difference_pp']],
                      on=['model','metric'],how='left')
report['status']=np.where(report['ours'].isna(),'not run','measured; review difference')
report.to_csv(RUN/'table1_final_report.csv',index=False)
display(report)
print('Saved report and logs:',RUN)


## 20 — Repeat training only after evaluation is understood

Open GitHub again, specifically `src/config/MACCO_CLIP.yaml`, `src/training/main_train_macco_clip.py`, `src/training/data_macco.py` and `scripts/train_MACCO_CLIP.sh`. The shell script alone is not a complete Colab setup.

Checklist before starting training:
1. Obtain the COCO images referenced by `mscoco_object_relation_attribute_with_phrase_bbox.tsv`. Inspect the TSV's columns and image paths; verify every referenced file. The provided annotations already contain phrases/bounding boxes; do not assume you need to rerun GroundingDINO for these annotations.
2. Use a separate training environment derived from upstream requirements, install its missing dependencies, and verify imports. This notebook's minimal evaluation environment is not a full training environment.
3. Match B/32 settings: seed 0, batch size 256, 5 epochs, warmup 50, CLIP learning rate 5e-7, predictor learning rate 1e-3, weight decay 0.2. Read the YAML for all loss weights and mask settings rather than copying only those six settings.
4. Put logs/checkpoints in Drive and correct all local dataset/evaluation paths. Validate one training batch as a diagnostic; do not report that tiny run as reproduction.
5. Use adequate GPU memory. The paper uses an A100. Reducing TRAINING batch size changes contrastive negatives and is a changed experiment; gradient accumulation is not automatically equivalent unless negatives are handled equivalently. Reducing evaluation batch size is different.
6. The inspected training code does not save optimizer state in its checkpoint and resets start_epoch to 0 when loading. Do not assume its resume option faithfully continues an interrupted run. Plan an uninterrupted 5-epoch run or explicitly implement and validate full training-state resume.
7. Run `sh scripts/train_MACCO_CLIP.sh` from the repository root only after the preceding checks. Evaluate the new checkpoint with the same evaluation pipeline; report it separately from released-weight results.

Table 15 describes four training seeds. Tables 9–12 require different training variants. Tables 7–8 use additional systems. These are later milestones, not part of the initial Table 1 evaluation. Exact training reproduction remains a separate task; the notebook does not launch unverified full training automatically.


## Troubleshooting in the order to check

| Symptom | Next check |
|---|---|
| `No module named open_clip` | Install `open_clip_torch`; restart; confirm version. |
| Torch/Torchvision import or operator error | Check that the installed pair is 2.6.0/0.21.0; restart after installation. |
| Checkpoint size/model mismatch | Confirm B/32 file and `ViT-B-32`; do not ignore missing weight keys. |
| Missing dataset/annotation | Inspect extracted folder layout and exact source path. |
| Missing image | Resolve the actual referenced image; do not skip it. |
| CUDA out of memory during evaluation | Lower copied evaluation batch size from 64 to 32/16 and record it; do not change precision first. |
| DataLoader crash/shared-memory problem | Lower copied workers from 2 to 0; record it. |
| spaCy/NLTK resource missing | Install `en_core_web_sm`, `punkt`, `punkt_tab` for ARO Order. |
| Score is 0.77 but paper is 77 | Convert fraction to percentage once. |
| Individual What's-up subset is unlike 43.2 | Use all three groups and the Table 20 aggregate. |
| Scores vary in ARO Order | Check shuffle seed, language resources and generated caption sets. |
| Colab disconnects | Recreate runtime setup and reload Drive artifacts; do not assume training resume is complete. |

## Source links

- [ACL paper](https://aclanthology.org/2026.acl-long.1490.pdf): Tables 1 and 15–20, Sections 3–4.
- [Official GitHub repository](https://github.com/hiker-lw/MACCO).
- [Main evaluation script](https://github.com/hiker-lw/MACCO/blob/97fbae1416e30eba84cc0b8209d9cb6bdedfe6ae/eval/evaluate_comp_benchmark.py).
- [ARO Order script](https://github.com/hiker-lw/MACCO/blob/97fbae1416e30eba84cc0b8209d9cb6bdedfe6ae/eval/evaluate_aro_order.py).
- [VL-Checklist script](https://github.com/hiker-lw/MACCO/blob/97fbae1416e30eba84cc0b8209d9cb6bdedfe6ae/eval/evaluate_vl_checklist.py).
- [Training config](https://github.com/hiker-lw/MACCO/blob/97fbae1416e30eba84cc0b8209d9cb6bdedfe6ae/src/config/MACCO_CLIP.yaml).
- [Released model files](https://huggingface.co/hiker-lw/MACCO/tree/main).
- [Benchmark archives](https://huggingface.co/datasets/hiker-lw/VL-Compositionality-Benchmarks/tree/main).
- [OpenCLIP 2.32.0 documentation](https://pypi.org/project/open-clip-torch/2.32.0/).
- [Colab FAQ](https://research.google.com/colaboratory/faq.html).

Upstream code is MIT licensed; this notebook uses and adapts its evaluation functions with attribution above. Colab adaptations and experiment reporting helpers are provided by this guide.
